In [ ]:
# Representation Scaling Experiments

Experimental pipeline for analyzing representation scaling in vision models under controlled conditions.

Evaluates how feature representations vary with:
- model depth
- dataset size

Comparisons are performed across architectures (ResNet, DenseNet, ViT).

Metrics:
- ℓ2 norm
- variance
- effective rank

In [ ]:
## Model Size Scaling

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os
import pandas as pd
import random
import numpy as np
import timm

# -------------------- Reproducibility --------------------
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -------------------- Config --------------------
DATA_FOLDER = "/content/images/200Images"
RESULTS_ROOT = "/content/results/model_ablation"

EPOCHS = 20
BATCH_SIZE = 1
LR = 0.001

MODEL_FAMILIES = {
    "resnet": ["resnet18", "resnet50", "resnet101"],
    "densenet": ["densenet121", "densenet169", "densenet201"],
    "vit": [
        "vit_small_patch16_224",
        "vit_base_patch16_224",
        "vit_large_patch16_224",
    ],
}

# -------------------- Transforms --------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# -------------------- Dataset --------------------
class SingleFolderDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.files = sorted(
            [f for f in os.listdir(folder_path)
             if f.lower().endswith((".png", ".jpg", ".jpeg"))],
            key=lambda x: int(os.path.splitext(x)[0])
        )
        self.folder_path = folder_path
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.folder_path, self.files[idx])
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.files[idx]

dataset = SingleFolderDataset(DATA_FOLDER, transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Images loaded:", len(dataset))

# -------------------- Model Factory --------------------
def get_model(name):
    if name.startswith("resnet"):
        model = getattr(models, name)(weights="DEFAULT")
        nf = model.fc.in_features
        model.fc = nn.Linear(nf, nf)
        head = model.fc

    elif name.startswith("densenet"):
        model = getattr(models, name)(weights="DEFAULT")
        nf = model.classifier.in_features
        model.classifier = nn.Linear(nf, nf)
        head = model.classifier

    elif name.startswith("vit"):
        model = timm.create_model(name, pretrained=True)
        nf = model.head.in_features
        model.head = nn.Linear(nf, nf)
        head = model.head

    else:
        raise ValueError(name)

    for p in model.parameters():
        p.requires_grad = False
    for p in head.parameters():
        p.requires_grad = True

    return model.to(device)

# -------------------- Rank Metric --------------------
def participation_ratio(h):
    sq = h.pow(2)
    return (sq.sum().pow(2) / (sq.pow(2).sum() + 1e-12)).item()

# -------------------- Model Ablation --------------------
criterion = nn.MSELoss()

for family, variants in MODEL_FAMILIES.items():
    family_dir = os.path.join(RESULTS_ROOT, family)
    os.makedirs(family_dir, exist_ok=True)

    print(f"\nRunning family: {family}")

    l2_table = {img: {} for img in dataset.files}
    var_table = {img: {} for img in dataset.files}
    rank_table = {img: {} for img in dataset.files}

    for model_name in variants:
        print(f"\nModel: {model_name}")
        model = get_model(model_name)

        optimizer = torch.optim.Adam(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR
        )

        # ---- Train projection head ----
        model.train()
        for epoch in range(EPOCHS):
            for x, _ in loader:
                x = x.to(device)
                optimizer.zero_grad()
                y = model(x)
                loss = criterion(y, y.detach())
                loss.backward()
                optimizer.step()

            print(f"[{family} | {model_name}] Epoch {epoch + 1}/{EPOCHS} completed")

        # ---- Collect metrics ----
        model.eval()
        with torch.no_grad():
            for x, name in loader:
                x = x.to(device)
                h = model(x).squeeze(0)

                l2_table[name[0]][model_name] = torch.norm(h, p=2).item()
                var_table[name[0]][model_name] = torch.var(h, unbiased=False).item()
                rank_table[name[0]][model_name] = participation_ratio(h)

    # ---- Save CSVs ----
    pd.DataFrame.from_dict(l2_table, orient="index") \
        .rename_axis("image_name") \
        .to_csv(os.path.join(family_dir, f"{family}_l2.csv"))

    pd.DataFrame.from_dict(var_table, orient="index") \
        .rename_axis("image_name") \
        .to_csv(os.path.join(family_dir, f"{family}_variance.csv"))

    pd.DataFrame.from_dict(rank_table, orient="index") \
        .rename_axis("image_name") \
        .to_csv(os.path.join(family_dir, f"{family}_rank.csv"))

    print(f"Saved: {family}_l2.csv, {family}_variance.csv, {family}_rank.csv")

print("\nModel ablation finished.")


Device: cuda
Images loaded: 200

Running family: resnet

Model: resnet18
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 202MB/s]


[resnet | resnet18] Epoch 1/20 completed
[resnet | resnet18] Epoch 2/20 completed
[resnet | resnet18] Epoch 3/20 completed
[resnet | resnet18] Epoch 4/20 completed
[resnet | resnet18] Epoch 5/20 completed
[resnet | resnet18] Epoch 6/20 completed
[resnet | resnet18] Epoch 7/20 completed
[resnet | resnet18] Epoch 8/20 completed
[resnet | resnet18] Epoch 9/20 completed
[resnet | resnet18] Epoch 10/20 completed
[resnet | resnet18] Epoch 11/20 completed
[resnet | resnet18] Epoch 12/20 completed
[resnet | resnet18] Epoch 13/20 completed
[resnet | resnet18] Epoch 14/20 completed
[resnet | resnet18] Epoch 15/20 completed
[resnet | resnet18] Epoch 16/20 completed
[resnet | resnet18] Epoch 17/20 completed
[resnet | resnet18] Epoch 18/20 completed
[resnet | resnet18] Epoch 19/20 completed
[resnet | resnet18] Epoch 20/20 completed

Model: resnet50
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 160MB/s]


[resnet | resnet50] Epoch 1/20 completed
[resnet | resnet50] Epoch 2/20 completed
[resnet | resnet50] Epoch 3/20 completed
[resnet | resnet50] Epoch 4/20 completed
[resnet | resnet50] Epoch 5/20 completed
[resnet | resnet50] Epoch 6/20 completed
[resnet | resnet50] Epoch 7/20 completed
[resnet | resnet50] Epoch 8/20 completed
[resnet | resnet50] Epoch 9/20 completed
[resnet | resnet50] Epoch 10/20 completed
[resnet | resnet50] Epoch 11/20 completed
[resnet | resnet50] Epoch 12/20 completed
[resnet | resnet50] Epoch 13/20 completed
[resnet | resnet50] Epoch 14/20 completed
[resnet | resnet50] Epoch 15/20 completed
[resnet | resnet50] Epoch 16/20 completed
[resnet | resnet50] Epoch 17/20 completed
[resnet | resnet50] Epoch 18/20 completed
[resnet | resnet50] Epoch 19/20 completed
[resnet | resnet50] Epoch 20/20 completed

Model: resnet101
Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


...

[vit | vit_small_patch16_224] Epoch 1/20 completed
[vit | vit_small_patch16_224] Epoch 2/20 completed
[vit | vit_small_patch16_224] Epoch 3/20 completed
[vit | vit_small_patch16_224] Epoch 4/20 completed
[vit | vit_small_patch16_224] Epoch 5/20 completed
[vit | vit_small_patch16_224] Epoch 6/20 completed
[vit | vit_small_patch16_224] Epoch 7/20 completed
[vit | vit_small_patch16_224] Epoch 8/20 completed
[vit | vit_small_patch16_224] Epoch 9/20 completed
[vit | vit_small_patch16_224] Epoch 10/20 completed
[vit | vit_small_patch16_224] Epoch 11/20 completed
[vit | vit_small_patch16_224] Epoch 12/20 completed
[vit | vit_small_patch16_224] Epoch 13/20 completed
[vit | vit_small_patch16_224] Epoch 14/20 completed
[vit | vit_small_patch16_224] Epoch 15/20 completed
[vit | vit_small_patch16_224] Epoch 16/20 completed
[vit | vit_small_patch16_224] Epoch 17/20 completed
[vit | vit_small_patch16_224] Epoch 18/20 completed
[vit | vit_small_patch16_224] Epoch 19/20 completed
[vit | vit_small_patc

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

[vit | vit_base_patch16_224] Epoch 1/20 completed
[vit | vit_base_patch16_224] Epoch 2/20 completed
[vit | vit_base_patch16_224] Epoch 3/20 completed
[vit | vit_base_patch16_224] Epoch 4/20 completed
[vit | vit_base_patch16_224] Epoch 5/20 completed
[vit | vit_base_patch16_224] Epoch 6/20 completed
[vit | vit_base_patch16_224] Epoch 7/20 completed
[vit | vit_base_patch16_224] Epoch 8/20 completed
[vit | vit_base_patch16_224] Epoch 9/20 completed
[vit | vit_base_patch16_224] Epoch 10/20 completed
[vit | vit_base_patch16_224] Epoch 11/20 completed
[vit | vit_base_patch16_224] Epoch 12/20 completed
[vit | vit_base_patch16_224] Epoch 13/20 completed
[vit | vit_base_patch16_224] Epoch 14/20 completed
[vit | vit_base_patch16_224] Epoch 15/20 completed
[vit | vit_base_patch16_224] Epoch 16/20 completed
[vit | vit_base_patch16_224] Epoch 17/20 completed
[vit | vit_base_patch16_224] Epoch 18/20 completed
[vit | vit_base_patch16_224] Epoch 19/20 completed
[vit | vit_base_patch16_224] Epoch 20/20

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

[vit | vit_large_patch16_224] Epoch 1/20 completed
[vit | vit_large_patch16_224] Epoch 2/20 completed
[vit | vit_large_patch16_224] Epoch 3/20 completed
[vit | vit_large_patch16_224] Epoch 4/20 completed
[vit | vit_large_patch16_224] Epoch 5/20 completed
[vit | vit_large_patch16_224] Epoch 6/20 completed
[vit | vit_large_patch16_224] Epoch 7/20 completed
[vit | vit_large_patch16_224] Epoch 8/20 completed
[vit | vit_large_patch16_224] Epoch 9/20 completed
[vit | vit_large_patch16_224] Epoch 10/20 completed
[vit | vit_large_patch16_224] Epoch 11/20 completed
[vit | vit_large_patch16_224] Epoch 12/20 completed
[vit | vit_large_patch16_224] Epoch 13/20 completed
[vit | vit_large_patch16_224] Epoch 14/20 completed
[vit | vit_large_patch16_224] Epoch 15/20 completed
[vit | vit_large_patch16_224] Epoch 16/20 completed
[vit | vit_large_patch16_224] Epoch 17/20 completed
[vit | vit_large_patch16_224] Epoch 18/20 completed
[vit | vit_large_patch16_224] Epoch 19/20 completed
[vit | vit_large_patc

In [ ]:
## Dataset Size Scaling

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import random
import numpy as np
import timm

# -------------------- Config --------------------
SEED = 0
EPOCHS = 20
BATCH_SIZE = 1
LR = 0.001

DATASETS = {
    100: "/content/images/100Images",
    200: "/content/images/200Images",
    300: "/content/images/300Images",
}

RESULTS_ROOT = "/content/results/dataset_ablation"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------- Reproducibility --------------------
def reset_seeds():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# -------------------- Transforms --------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# -------------------- Dataset --------------------
class SingleFolderDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.files = sorted(
            [f for f in os.listdir(folder_path)
             if f.lower().endswith((".png", ".jpg", ".jpeg"))],
            key=lambda x: int(os.path.splitext(x)[0])
        )
        self.folder_path = folder_path
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.folder_path, self.files[idx])
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.files[idx]

# -------------------- Models --------------------
def get_resnet18():
    reset_seeds()
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    nf = model.fc.in_features
    model.fc = nn.Linear(nf, nf)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.fc.parameters():
        p.requires_grad = True
    return model.to(device)

def get_densenet121():
    reset_seeds()
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    nf = model.classifier.in_features
    model.classifier = nn.Linear(nf, nf)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True
    return model.to(device)

def get_vit_small():
    reset_seeds()
    model = timm.create_model("vit_small_patch16_224", pretrained=True)
    nf = model.head.in_features
    model.head = nn.Linear(nf, nf)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True
    return model.to(device)

MODELS = {
    "resnet18": get_resnet18,
    "densenet121": get_densenet121,
    "vit_small": get_vit_small,
}

criterion = nn.MSELoss()

# -------------------- Rank Metric --------------------
def participation_ratio(h):
    sq = h.pow(2)
    return (sq.sum().pow(2) / (sq.pow(2).sum() + 1e-12)).item()

# -------------------- Dataset-Size Ablation --------------------
for model_name, model_fn in MODELS.items():
    print(f"\nRunning dataset ablation for {model_name}")
    model_dir = os.path.join(RESULTS_ROOT, model_name)
    os.makedirs(model_dir, exist_ok=True)

    l2_table = {}
    var_table = {}
    rank_table = {}

    for size, folder in DATASETS.items():
        print(f"  Dataset size: {size}")
        dataset = SingleFolderDataset(folder, transform)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

        model = model_fn()
        optimizer = torch.optim.Adam(
            [p for p in model.parameters() if p.requires_grad], lr=LR
        )

        model.train()
        for epoch in range(EPOCHS):
            for x, _ in loader:
                x = x.to(device)
                optimizer.zero_grad()
                y = model(x)
                loss = criterion(y, y.detach())
                loss.backward()
                optimizer.step()
            print(f"    [{model_name}] Epoch {epoch+1}/{EPOCHS} completed")

        model.eval()
        with torch.no_grad():
            for x, name in loader:
                x = x.to(device)
                h = model(x).squeeze(0)

                l2_table.setdefault(name[0], {})[size] = torch.norm(h, p=2).item()
                var_table.setdefault(name[0], {})[size] = torch.var(h, unbiased=False).item()
                rank_table.setdefault(name[0], {})[size] = participation_ratio(h)

    # ---- Save CSVs ----
    pd.DataFrame.from_dict(l2_table, orient="index") \
        .rename_axis("image_name") \
        .to_csv(os.path.join(model_dir, f"{model_name}_l2.csv"))

    pd.DataFrame.from_dict(var_table, orient="index") \
        .rename_axis("image_name") \
        .to_csv(os.path.join(model_dir, f"{model_name}_variance.csv"))

    pd.DataFrame.from_dict(rank_table, orient="index") \
        .rename_axis("image_name") \
        .to_csv(os.path.join(model_dir, f"{model_name}_rank.csv"))

    print(f"Saved {model_name}_l2.csv, {model_name}_variance.csv, {model_name}_rank.csv")

print("\nDataset-size ablation finished.")


Using device: cuda

Running dataset ablation for resnet18
  Dataset size: 100
    [resnet18] Epoch 1/20 completed
    [resnet18] Epoch 2/20 completed
    [resnet18] Epoch 3/20 completed
    [resnet18] Epoch 4/20 completed
    [resnet18] Epoch 5/20 completed
    [resnet18] Epoch 6/20 completed
    [resnet18] Epoch 7/20 completed
    [resnet18] Epoch 8/20 completed
    [resnet18] Epoch 9/20 completed
    [resnet18] Epoch 10/20 completed
    [resnet18] Epoch 11/20 completed
    [resnet18] Epoch 12/20 completed
    [resnet18] Epoch 13/20 completed
    [resnet18] Epoch 14/20 completed
    [resnet18] Epoch 15/20 completed
    [resnet18] Epoch 16/20 completed
    [resnet18] Epoch 17/20 completed
    [resnet18] Epoch 18/20 completed
    [resnet18] Epoch 19/20 completed
    [resnet18] Epoch 20/20 completed
  Dataset size: 200
    [resnet18] Epoch 1/20 completed
    [resnet18] Epoch 2/20 completed
    [resnet18] Epoch 3/20 completed
    [resnet18] Epoch 4/20 completed
    [resnet18] Epoch 5/20 c

In [ ]:
## Notes

- Experiments use frozen backbones under controlled settings
- Results correspond to the associated research report

## Reproducibility Note

This code was developed and executed in a controlled Colab environment.
Paths may need to be adjusted based on local directory structure before running.